# Trabajo final · Data Scientist · Tonada

**App de música con plan gratuito y premium** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿A qué usuarios gratuitos conviene ofrecerles un descuento para que pasen a premium?**

**Decisión que habilita:** A quién se le ofrece el descuento del 20 %.

**Tipo de problema:** Clasificación y modelos de uplift (efecto incremental).

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNq9Wcty3MYV3esrupCiC5CHEGaGpCwqdFVCWxUnpuSSJwsHNTXVBBpDWAAawYMmpdD/kqUXWekT+GM593bjNRzRyiYscqYft2/f"
    "vs/TzTQvddWIos3LWyFrUZQzUcoiRhO/ZfzkSawSsVWFqmS1iWWja7dWeZpl8mwRLE6OZuJaVbGMz17JrFbe6ROBn0qcgZNfgZHO"
    "fXCQbdZsqmLbrfVeitegWQT4eSl+ePvt+XdvqH8cBIaB2qJbxv4qzVXdyLx0HWx3dBjM8et44kuabPSmwXysska6lZ8WjdqqqnaD"
    "mTgJ8PHam4m2SJsz5xsHOyqIaQSLsrTEgkJXuczc5Xwm5pZ8fjwTzwPPr3RbxC4WxWldaqyq/OhKp5FyQwenqnQab7Yyl5tL+bN0"
    "ZqIbpGaqa/r6RV06a3CFQs9CfwFN+csTfCzoY3609vikZaWvp+xXbdTm9/8uiMf5/ccq1pe8w48Sx6TGhYJa33Pzr+3P7S01/tyq"
    "QtfiT2mlakNcNFK8UsxEQoGyiuRInPmShPiKPhb0gb+AegvucfdkjePD8puyggtMREx09i7TFXOPWlk1qtHUbmSx5Ualo3ebQkap"
    "LmRmqPLLlEWOriBMzktLXW4y2aSFHiuKJTjqBaLeCcszVpsyhlQ3Izuin+mte+Qfw5CB/9UxmRRnuNSNoX6vKrgv9rnUOjMTYX8q"
    "DC+wolJlJiNl3XmNhauqVXCeENTUfWE8tFZFbbj+cqUq5brWUc7EHv/wxL+Ey+73R2xCbubD3/q1w1LjOoFPkgT+8bHniacDHXP4"
    "Whzx5AlzMdr4g9CJqhpZc+8Cci1POLLatPN4XUSyUQX+3BB9idjc4tCQZayBC3FIdthRAlRV+fVVmySZcsHSbJrofSF6fBgsPy9E"
    "F8/xcTENUcvyG3jsqwpe4n5wWueUjgF3SdBK9J3n10hZm2uZtaqGM7YOz60RtKpWzSYtYnXjwgTlGZnOCKtDB2587ZAFtb9FeJeX"
    "ty7Wen6Co6VJ485p/9CJU1lvelpX+wl0on0a8fy48WN5W7+E+bfMqaf2E+S1QrovXrzwfCMbjH1inKVl2taOw5nonCps1y8FDNNU"
    "OuP4MvnSvfCwEtYPugRRbij0QTIXz4Q7h2at6x92Pq+0Bylt5zl7zdw/IuePTSrtfWi6y8CcFsxHzhAuYJ1lgLi8MHFJXgePXHuz"
    "TzELmIYyaeAZrcfjne1Jx3uQF/AWYEqCkrwUZ9CM+EL8alesuVIYZQR7tXC49F+gj3gw4WJV4pECysByJDaBb/hsh/SxX4eURKDF"
    "p2JBmy3AlsKRB4hpI1vmNn+OmS3+KB2wPc1Mf2j4yUwcYlskNEx5DwiMbFCZmTfJrY4MSa0yFTVuGFNyAAV/z7vGomuwmUiZsAB9"
    "PD/m3EBxW44KHpT3JW//lHYgQl7y4thjN7x+4INlFzoxx0LM8WGNwiO23Y1fd4PXPFIG3C8D7mFj7uKb2UYmgVJ8fb0vjcznh8uA"
    "cgJTp8UW0a13/Ol61mGHp+QRh1DHM+gn4PwfYG1iz0W7/F4+Wu6mI3N6k1gf5iUkeDOHrDQk1DnlUHiiRzgg3rQ1qmOqKYXRKCUq"
    "hQI4rIRgNitgLoZZUMchYxk1mKSkF0lU0IF+VIRVLlOurWVbXzHuKDayLJ1RyBLkoLBdcNg626ot9aYz4KnoI5JyI2r1Jktr3sUo"
    "lXaH+tKqwdQyiM2Ka1/WzW2p3CTTsuFzGtsgDUKz1QZmBKUd7E9MK6FoAAJMQpU/qipF+mYDeROLYvK1XHVJ9K6rcKhJlY7biDCF"
    "qk/FEWIuh3IAVGHEWsRKZFIYGaQ1Gw1GMpbC2sEm1DTfVwXytCDAl9h5+kJFMdVkXDG9Po0XqJK3NYdNqdO61oWr4IlLr1MR3Mvr"
    "yDr80EHc1qYBnEvJxp3WZLMEa1cgylThtpw6BjdU5MXfsWwscNi2a0pfuy4+pOgVFYXFHB9fnRwhRDoZsTHEPDnqPb+2nn+lK8CX"
    "DjuNshWlq51SfgS2J2isKPI6SfHxUKBgN8Ber4xSAgM2gVL+H1hz9ZlgE5lRAlLENxDqw3ZzOqiiF/ZMbDdeGKyBTSo04fmCAIqK"
    "kfybjszz7vi0MsH0KM+uTOF8DpXBm8eqJkpk9V4TVo6wQ+ThzRoWpz1vaMu2DU8D1Lg0EYzahKIPrsW88cBe5WVz60IBMdn/zPio"
    "xfg9cxjZgjw600yQBrCLFcJPwa52vVNBkeKaxawHlG0S3BwnzNdjW2LhTOR+3eauNwgV/koLDOHYp8hELs95dgnAQVttminl3OAU"
    "XmCY1u/S8oGKXdTqY6RgfFGtUHRAAuPEFTV605dcW7bBY+LkSN3Hxr1Bg0yR5hCJxQnpENOIOOlEWnn2oHu26ENpQZSWwLO333Gq"
    "21t6RoWlqywYJVEwxLYelxsKZRqnIoOdcLXVNW3XYhJNTNn9Ma+rFF4wLTWUjag2UCBdtvU/W9xEOPRknJoYVJHOcS0FNwqv1VB/"
    "5qb+0Mfas8kcXiQBsvTek9kzTEoqcoStqfwSQce2cUU3b9Sneqc4Jt3o5sO702AZ3znsye/Ihw1X2MfjREBcYUjOHxuogeou2dXK"
    "ahX9e1aYittJa7QPYAtRWOqK4KBD9zh06Is2p9tfnTbpNVFwjyuyvk6LKKVzUZuGMokKHzWtpOKNai6bNkUy7Evk4eGhqNsoJcZW"
    "eIjdnQB3wPKWKlxbA+JGYR8gr00OIvxjRFsbPy1k8UnigADtQD2B9HPihHsQ1DunxGhFCXfVsd57L7EbwGOw9XQFXb/wnVA1cZ2D"
    "+NlB/uzgJ8f7fdKfDg/yw4PY8fj44Ui7jwpBRYHQ85Ws/LYsVcVlupZVJW9d7Nmz6ZIptvRMjniMSl/+DFhvU0PFyWoS8b2l4pYn"
    "cREALsbdO4GbnkGs5UwYQTdw8YZzeP+yRpf10nirufS7YYVcBlZUzZr349RIyKIqPWv/YytPOM4aE/0072ckzTC9X82rg7+cHlyc"
    "HvzIrwDOP5Ah3MkygwtWPSi40m1Vny09bz8/MfDrTrejk/kjCuFgfOxtIKKS0uWkXvdbujd8GODHqQidVyMsgnbUtccY5dWb7//2"
    "/Zu33wpZIUd1gGMHpRCztxgRr0e4hUjEGMi8tQP+Q5izHqMh4nY+gkZjmHT+9z+9XX27emNXMDIy9D1I6ltmTFxDbari10KLsmjB"
    "qsNbZmg9glfMj9v3v+3ArnPbXE8hGK34QZfCdqeT6NHc9xat3VkjhV3yJ5cccNH2GkjII9zTu7Xx5xMDgW4GjBQ1vmGxtlc7ugaY"
    "+11vdm1iTie7Mbd43MUWtDaZxp1OZsxvzXPh9O63P/FQROrE63Mg5Biv+nQGfJxwyH/25OHOZfMzpKGsg10m68YXna5NOZDC/sD5"
    "fPqJgkZXxbUxxe74p05nHxZis6q/oftpjbud9jn6Q3p4WHdPdJQbgFk/ceQXLBcXQHAFGt+5EO+//K6H+mneLvJyN+kOSg2W/RZE"
    "98Xe4xa6KaTr7btM017mXyPzBb27vuzc+n/Jj8tH8yP/14eTYQclCHbWDPvGZYvwTclPFiaV0nsBATQbYvTakdxZgNu0VWEYP3ny"
    "zXfn57vsd+AVepRLkxS3eUQzqkYHa5x9MMuJ7//Dl/9+bEBd/I25KANgMzB2isDGXXoFQOUuTT4eY7JR5yE0o273IAE7IWMLmYmc"
    "IHKj7YRNA3cz1gj9PFTnrhZGZx6QssNwv0fzzi7qd6b7Dpvcf+T/Me2/EPTDAmjfcr//6IwvCJRCwDMGH3zhWqne338UvSjV5Gwj"
    "pxjD/D2G5Zkx1He2979RczI7oP6++RDI2z4E7NcKVYjuaBMBBy+dvu09kHBkvN1Hvs5ArDPWdZRepkZtOw9/vYtOuO2+AA4DICJu"
    "Me4GgahTUWgcww5C64V15Z3nQpTVqjMQyHFJa4s0MgLtvgY64Dpy2ARHkHW6Jc7kvfK9pNMTNYndFkLdgDZl33IeviHaPh+RhgQI"
    "oZ2MLcEBAoI8bXNnzzsjyVLKmt3Jko32zuCVy0CQBmvnUw+Q3ajZj2cgQg2220mUTB4mjVXgImTAHQGcu7sn/wUVK7vh"
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "04_Tonada"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Quitar reproducciones duplicadas y saltos de menos de 5 segundos, detectar bots y cuentas compartidas, y unificar géneros.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Conversión con y sin descuento, su diferencia con intervalo y una prueba de hipótesis sobre el efecto.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Propensión con regresión logística y XGBoost, modelos de uplift con los vistos y K-means de contraste; evaluación en el grupo de control.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red contra los modelos de árboles: ¿mejora el uplift o solo la propensión?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con uplift por decil, costo de los descuentos e ingreso incremental.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada, versión y por qué hay que conservar siempre un grupo de control.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** La política de descuentos según margen: a quién ofrecer el 20 % y a quién no ofrecer nada.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**